<a href="https://colab.research.google.com/github/321222/CLaude/blob/main/Automate_data_vaildation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automated Data Validation & Reporting Tool

This pipeline automates the ingestion, validation, cleaning, and reporting of tabular data (CSV/Excel). It checks for:
- Missing values
- Duplicate rows
- Incorrect data types
- Out-of-range numerical values
- Inconsistent categorical values (e.g., casing issues)

It logs all detected issues, performs safe fixes, generates an interactive validation scorecard, and exports a clean dataset alongside a multi-tab Excel validation report.

### 1. Project README & Git Commit Guidelines

```markdown
# Data Quality Pipeline

An automated Python pipeline for validating, cleaning, and reporting data issues.

## Structure
- `validator.py`: Contains core classes (`DataValidator`, `DataCleaner`, `ReportGenerator`).
- `test_validator.py`: Unit tests verifying cleaning and validation logic.
- `main.py`: Main entry point demonstrating pipeline execution on sample data.

## Setup & Execution
1. Install dependencies:
   ```bash
   pip install pandas openpyxl jinja2
   ```
2. Run the pipeline:
   ```bash
   python main.py
   ```

## Suggested Git Commits
- `feat: implement DataValidator engine with missing, duplicate, and range checks`
- `feat: add DataCleaner module for auto-correcting types and casing`
- `feat: implement ReportGenerator to export multi-tab Excel validation summary`
- `test: add unit tests for validator and cleaner functions`
- `docs: add installation and usage instructions to README`
```

### 2. Core Python Pipeline Code (Validation, Cleaning & Reporting)

In [1]:
import os
import logging
import pandas as pd
import numpy as np

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("DataPipeline")

class DataValidator:
    def __init__(self, df: pd.DataFrame, schema: dict):
        """
        schema structure:
        {
            'types': {'col_name': 'type_str'},
            'ranges': {'col_name': (min_val, max_val)},
            'allowed_values': {'col_name': [val1, val2]}
        }
        """
        self.df = df.copy()
        self.schema = schema
        self.logs = []

    def validate(self) -> dict:
        self.logs = []
        total_rows = len(self.df)

        # 1. Missing values check
        missing_counts = self.df.isnull().sum()
        for col, count in missing_counts.items():
            if count > 0:
                self.logs.append({
                    'check': 'missing_values',
                    'column': col,
                    'severity': 'Warning',
                    'message': f"{count} missing value(s) found ({(count/total_rows)*100:.1f}% of total)."
                })

        # 2. Duplicates check
        dup_count = self.df.duplicated().sum()
        if dup_count > 0:
            self.logs.append({
                'check': 'duplicate_rows',
                'column': 'ALL',
                'severity': 'Warning',
                'message': f"{dup_count} duplicate row(s) found in the dataset."
            })

        # 3. Data type mismatch check
        for col, expected_type in self.schema.get('types', {}).items():
            if col in self.df.columns:
                actual_type = str(self.df[col].dtype)
                # Basic mapping checking
                if expected_type == 'int' and not pd.api.types.is_integer_dtype(self.df[col]):
                    self.logs.append({
                        'check': 'type_mismatch',
                        'column': col,
                        'severity': 'Error',
                        'message': f"Expected '{expected_type}', found type '{actual_type}'."
                    })
                elif expected_type == 'float' and not pd.api.types.is_numeric_dtype(self.df[col]):
                    self.logs.append({
                        'check': 'type_mismatch',
                        'column': col,
                        'severity': 'Error',
                        'message': f"Expected '{expected_type}', found type '{actual_type}'."
                    })

        # 4. Out of range values
        for col, (min_val, max_val) in self.schema.get('ranges', {}).items():
            if col in self.df.columns and pd.api.types.is_numeric_dtype(self.df[col]):
                out_of_range = self.df[(self.df[col] < min_val) | (self.df[col] > max_val)]
                if not out_of_range.empty:
                    self.logs.append({
                        'check': 'out_of_range',
                        'column': col,
                        'severity': 'Warning',
                        'message': f"{len(out_of_range)} value(s) out of acceptable range [{min_val}, {max_val}]."
                    })

        return {
            'scorecard': self.calculate_score(),
            'issues': pd.DataFrame(self.logs)
        }

    def calculate_score(self) -> float:
        if not self.logs:
            return 100.0
        # Errors deduct 15 points, Warnings deduct 5 points
        deductions = sum(15 if log['severity'] == 'Error' else 5 for log in self.logs)
        return max(0.0, 100.0 - deductions)


class DataCleaner:
    @staticmethod
    def clean(df: pd.DataFrame, schema: dict) -> pd.DataFrame:
        df_clean = df.copy()
        logger.info("Starting data cleaning process...")

        # 1. Drop complete duplicates
        dup_count = df_clean.duplicated().sum()
        if dup_count > 0:
            df_clean = df_clean.drop_duplicates().reset_index(drop=True)
            logger.info(f"Removed {dup_count} duplicate rows.")

        # 2. Correct string formatting & Casing issues
        for col in df_clean.select_dtypes(include=['object']).columns:
            df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

        # 3. Handle data types safely with coercion
        for col, expected_type in schema.get('types', {}).items():
            if col in df_clean.columns:
                if expected_type == 'int':
                    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(int)
                elif expected_type == 'float':
                    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

        logger.info("Data cleaning completed successfully.")
        return df_clean


class ReportGenerator:
    @staticmethod
    def generate_excel_report(original_df: pd.DataFrame, clean_df: pd.DataFrame, issues_df: pd.DataFrame, score: float, filepath: str):
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            # Sheet 1: Scorecard & Overview
            summary_data = {
                'Metric': ['Overall Data Quality Score', 'Original Row Count', 'Cleaned Row Count', 'Total Issues Found'],
                'Value': [f"{score}%", len(original_df), len(clean_df), len(issues_df)]
            }
            pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary Scorecard', index=False)

            # Sheet 2: Validation Issues Logs
            if not issues_df.empty:
                issues_df.to_excel(writer, sheet_name='Issues Log', index=False)
            else:
                pd.DataFrame([{'Status': 'All checks passed! No issues detected.'}]).to_excel(writer, sheet_name='Issues Log', index=False)

            # Sheet 3: Cleaned Dataset
            clean_df.to_excel(writer, sheet_name='Clean Data', index=False)

        logger.info(f"Excel report successfully compiled and saved to '{filepath}'")

### 3. Unit Tests (Validating Code Correctness)

In [4]:
import unittest

class TestValidationAndCleaning(unittest.TestCase):
    def setUp(self):
        self.test_schema = {
            'types': {'Age': 'int', 'Salary': 'float'},
            'ranges': {'Age': (18, 65)}
        }
        self.raw_data = pd.DataFrame({
            'Name': [' alice ', 'bob', 'bob'],
            'Age': [25, 150, 150],  # Age has been aligned to test duplicate detection
            'Salary': ['50000', 'invalid_salary', 'invalid_salary']  # Salary has been aligned to test duplicate detection
        })

    def test_cleaner_removes_duplicates_and_formats_strings(self):
        cleaned_df = DataCleaner.clean(self.raw_data, self.test_schema)
        # Bob's duplicate row should be removed
        self.assertEqual(len(cleaned_df), 2)
        # Name should be properly titled and stripped
        self.assertEqual(cleaned_df.loc[0, 'Name'], 'Alice')

    def test_validator_identifies_out_of_range(self):
        validator = DataValidator(self.raw_data, self.test_schema)
        results = validator.validate()
        issues = results['issues']

        # Check if the out-of-range value (Age=150) was flagged
        out_of_range_issues = issues[issues['check'] == 'out_of_range']
        self.assertFalse(out_of_range_issues.empty)
        self.assertEqual(out_of_range_issues.iloc[0]['column'], 'Age')

# Run the tests programmatically in Jupyter/Colab environment
suite = unittest.TestLoader().loadTestsFromTestCase(TestValidationAndCleaning)
unittest.TextTestRunner(verbosity=2).run(suite)

test_cleaner_removes_duplicates_and_formats_strings (__main__.TestValidationAndCleaning.test_cleaner_removes_duplicates_and_formats_strings) ... ok
test_validator_identifies_out_of_range (__main__.TestValidationAndCleaning.test_validator_identifies_out_of_range) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.023s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### 4. Running the Pipeline on Demonstration Data

In [3]:
# 1. Define schema requirements
schema_config = {
    'types': {
        'Age': 'int',
        'PerformanceScore': 'float'
    },
    'ranges': {
        'Age': (18, 100),
        'PerformanceScore': (1.0, 5.0)
    }
}

# 2. Generate simulated raw input file (with intentional issues)
simulated_data = pd.DataFrame({
    'EmployeeName': [' john doe ', 'Jane Smith', 'Jane Smith', 'Invalid User', 'Mark Taylor'],
    'Age': [34, 28, 28, 150, np.nan],  # Jane is duplicated, Invalid User is out of age range, Mark has missing age
    'PerformanceScore': [4.5, 4.8, 4.8, 3.2, 5.5]  # Mark's score is out-of-bounds
})
simulated_data.to_csv('employee_raw_data.csv', index=False)

# 3. Read and execute pipeline workflow
logger.info("Loading raw data for pipeline processing...")
input_df = pd.read_csv('employee_raw_data.csv')

# Validate Raw Data
validator = DataValidator(input_df, schema_config)
validation_results = validator.validate()
raw_issues = validation_results['issues']
quality_score = validation_results['scorecard']

# Clean Data
cleaned_df = DataCleaner.clean(input_df, schema_config)

# Export report and clean output
report_path = 'data_quality_report.xlsx'
ReportGenerator.generate_excel_report(input_df, cleaned_df, raw_issues, quality_score, report_path)

# Print scorecard summary details
print("\n" + "="*40)
print(f"DATA QUALITY SCORECARD: {quality_score} / 100")
print("="*40)
display(raw_issues)

print("\nFinal Cleaned DataFrame Sample:")
display(cleaned_df)


DATA QUALITY SCORECARD: 65.0 / 100


,check,column,severity,message
0,missing_values,Age,Warning,1 missing value(s) found (20.0% of total).
1,duplicate_rows,ALL,Warning,1 duplicate row(s) found in the dataset.
2,type_mismatch,Age,Error,"Expected 'int', found type 'float64'."
3,out_of_range,Age,Warning,"1 value(s) out of acceptable range [18, 100]."
4,out_of_range,PerformanceScore,Warning,"1 value(s) out of acceptable range [1.0, 5.0]."



Final Cleaned DataFrame Sample:


,EmployeeName,Age,PerformanceScore
0,John Doe,34,4.5
1,Jane Smith,28,4.8
2,Invalid User,150,3.2
3,Mark Taylor,0,5.5
